# 02 · Classical baselines — TF-IDF + Logistic Regression / Linear SVM / Random Forest

Trains the three classical models of the paper (Table 3) on the frozen splits.

* Hyper-parameters tuned on **validation macro-F1 only** (search spaces from paper Table 4).
* Each model is evaluated under two conditions: `baseline` and `class_weight` (cost-sensitive loss).
* Logs metrics to `results/model_metrics.csv`, latency to `results/latency_results.csv`, predictions to `results/predictions_<model>_<condition>.csv`.
* Saves fitted pipelines to `models/classical/`.

**Needs:** `data/splits/*.csv` from notebook 01.

In [ ]:
!pip -q install scikit-learn seaborn joblib

In [ ]:
import os, sys, json, random, time, warnings, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# ---- Project root: Google Drive folder on Colab, repo root locally ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/mental-health-nlp")   # <- change if you keep the repo elsewhere
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for d in ["data/raw", "data/interim", "data/processed", "data/splits",
          "models/classical", "models/bilstm", "models/distilbert", "models/roberta",
          "results/figures", "results/reports"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SEED = 42
EMOTIONS = ["Joy", "Sadness", "Anger", "Fear", "Anxiety", "Stress", "Surprise", "Neutral"]
LABEL2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2LABEL = {i: e for e, i in LABEL2ID.items()}
NEGATIVE = ["Sadness", "Anger", "Fear", "Anxiety", "Stress"]      # N in the paper (Sec. 5.6)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()
print("ROOT =", ROOT)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             confusion_matrix, classification_report)

METRICS_CSV = ROOT / "results/model_metrics.csv"
LATENCY_CSV = ROOT / "results/latency_results.csv"
LABEL_IDS = list(range(len(EMOTIONS)))

def slug(condition):
    """File-system friendly condition name ('augment+class_weight' -> 'augment-class_weight')."""
    return condition.replace("+", "-")

def _upsert(path, row, keys):
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(df):
        mask = np.ones(len(df), dtype=bool)
        for k in keys:
            mask &= (df[k].astype(str) == str(row[k])).values
        df = df[~mask]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)

def evaluate(y_true, y_pred, model, condition="baseline", seed=SEED, extra=None, log=True):
    """Compute all metrics, optionally upsert into results/model_metrics.csv."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABEL_IDS,
                                                 average="macro", zero_division=0)
    per = f1_score(y_true, y_pred, labels=LABEL_IDS, average=None, zero_division=0)
    row = dict(model=model, condition=condition, seed=seed,
               accuracy=accuracy_score(y_true, y_pred),
               precision_macro=p, recall_macro=r, macro_f1=f, min_class_f1=per.min())
    row.update({f"f1_{e}": v for e, v in zip(EMOTIONS, per)})
    if extra:
        row.update(extra)
    if log:
        _upsert(METRICS_CSV, row, ["model", "condition", "seed"])
    return row

def log_latency(model, device, ms_per_text, note=""):
    _upsert(LATENCY_CSV, dict(model=model, device=device, ms_per_text=round(float(ms_per_text), 2), note=note),
            ["model", "device", "note"])

def plot_cm(y_true, y_pred, title, path=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)
    if normalize:
        cm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax, cbar=False)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()

In [ ]:
SPLIT_DIR = ROOT / "data/splits"
train = pd.read_csv(SPLIT_DIR / "train.csv")
val   = pd.read_csv(SPLIT_DIR / "val.csv")
test  = pd.read_csv(SPLIT_DIR / "test.csv")
for d in (train, val, test):
    d["text_tf"] = d["text_tf"].fillna("").astype(str)
    d["text_clean"] = d["text_clean"].fillna("").astype(str)
print({k: len(v) for k, v in dict(train=train, val=val, test=test).items()})
train["label"].value_counts().reindex(EMOTIONS)

## 1 · Hyper-parameter search (validation set only)

In [ ]:
import itertools, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

QUICK = True     # True: reduced grids so the notebook finishes in minutes. False: full paper grid.

X_tr, y_tr = train["text_clean"], train["label_id"].values
X_va, y_va = val["text_clean"],   val["label_id"].values
X_te, y_te = test["text_clean"],  test["label_id"].values

NGRAMS   = [(1, 1), (1, 2)]
MAXFEATS = [20000, 50000]
C_GRID   = [0.1, 1, 10] if QUICK else [0.01, 0.1, 1, 10]
RF_TREES = [200] if QUICK else [200, 500]
RF_DEPTH = [None, 30] if QUICK else [None, 30, 60]
RF_LEAF  = [1] if QUICK else [1, 2]

def make_clf(name, p, cw):
    if name == "logreg":
        return LogisticRegression(C=p["C"], max_iter=1000, class_weight=cw)
    if name == "svm":
        return LinearSVC(C=p["C"], class_weight=cw)
    if name == "rf":
        return RandomForestClassifier(n_estimators=p["n"], max_depth=p["depth"], min_samples_leaf=p["leaf"],
                                      class_weight="balanced_subsample" if cw else None,
                                      n_jobs=-1, random_state=SEED)

def make_vec(p):
    return TfidfVectorizer(ngram_range=p["ngram"], max_features=p["maxfeat"], min_df=2)

def tune_linear(name):
    best = dict(f1=-1)
    for ng, mf in itertools.product(NGRAMS, MAXFEATS):
        vec = make_vec(dict(ngram=ng, maxfeat=mf))
        Xt, Xv = vec.fit_transform(X_tr), vec.transform(X_va)
        for C in C_GRID:
            clf = make_clf(name, dict(C=C), None).fit(Xt, y_tr)
            f1 = f1_score(y_va, clf.predict(Xv), average="macro")
            print(f"{name:7s} ngram={ng} maxfeat={mf:>5} C={C:<5} val macro-F1={f1:.4f}")
            if f1 > best["f1"]:
                best = dict(f1=f1, ngram=ng, maxfeat=mf, C=C)
    return best

BEST = {}
for nm in ["logreg", "svm"]:
    BEST[nm] = tune_linear(nm)
    print("-> best", nm, BEST[nm], "\n")

In [ ]:
# Random Forest: reuse the vectoriser settings that won for Logistic Regression (keeps the RF grid affordable)
vec_p = dict(ngram=BEST["logreg"]["ngram"], maxfeat=BEST["logreg"]["maxfeat"])
vec = make_vec(vec_p); Xt, Xv = vec.fit_transform(X_tr), vec.transform(X_va)
best_rf = dict(f1=-1)
for n, d, l in itertools.product(RF_TREES, RF_DEPTH, RF_LEAF):
    clf = make_clf("rf", dict(n=n, depth=d, leaf=l), None).fit(Xt, y_tr)
    f1 = f1_score(y_va, clf.predict(Xv), average="macro")
    print(f"rf trees={n} depth={d} leaf={l} val macro-F1={f1:.4f}")
    if f1 > best_rf["f1"]:
        best_rf = dict(f1=f1, n=n, depth=d, leaf=l, **vec_p)
BEST["rf"] = best_rf
print("-> best rf", best_rf)
json.dump({k: {kk: (list(vv) if isinstance(vv, tuple) else vv) for kk, vv in v.items()} for k, v in BEST.items()},
          open(ROOT / "models/classical/best_params.json", "w"), indent=2)

## 2 · Final training & test evaluation
Both conditions reuse the hyper-parameters tuned on the *baseline*, so the comparison isolates the effect of class weighting.

In [ ]:
import time
PIPES = {}

def build_pipeline(name, cw):
    p = BEST[name]
    return Pipeline([("tfidf", make_vec(p)), ("clf", make_clf(name, p, cw))])

def time_single(pipe, text, n=200):
    pipe.predict([text])  # warm-up
    t0 = time.perf_counter()
    for _ in range(n):
        pipe.predict([text])
    return 1000 * (time.perf_counter() - t0) / n

sample_text = test["text_clean"].iloc[0]
for name in ["logreg", "svm", "rf"]:
    for cond, cw in [("baseline", None), ("class_weight", "balanced")]:
        t0 = time.time()
        pipe = build_pipeline(name, cw).fit(X_tr, y_tr)
        val_f1 = f1_score(y_va, pipe.predict(X_va), average="macro")
        pred = pipe.predict(X_te)
        row = evaluate(y_te, pred, model=name, condition=cond, extra=dict(val_macro_f1=val_f1))
        PIPES[(name, cond)] = pipe
        joblib.dump(pipe, ROOT / f"models/classical/{name}_{cond}.joblib")

        out = pd.DataFrame({"id": test["id"], "source": test["source"], "text": test["text"],
                            "true": [ID2LABEL[i] for i in y_te], "pred": [ID2LABEL[i] for i in pred]})
        if hasattr(pipe, "predict_proba"):
            out["conf"] = pipe.predict_proba(X_te).max(axis=1)
        out.to_csv(ROOT / f"results/predictions_{name}_{cond}.csv", index=False)

        if cond == "baseline":
            log_latency(name, "cpu", time_single(pipe, sample_text), note="inference only (classical features), batch=1")
        print(f"{name:7s} {cond:13s} acc={row['accuracy']:.4f} macro-F1={row['macro_f1']:.4f} "
              f"min-class-F1={row['min_class_f1']:.4f}  ({time.time()-t0:.0f}s)")

## 3 · Results

In [ ]:
m = pd.read_csv(METRICS_CSV)
cl = m[m.model.isin(["logreg", "svm", "rf"])]
cols = ["model", "condition", "accuracy", "precision_macro", "recall_macro", "macro_f1", "min_class_f1"]
display(cl[cols].round(4).sort_values("macro_f1", ascending=False))

print("\nPer-class F1 (test):")
display(cl.set_index(["model", "condition"])[[f"f1_{e}" for e in EMOTIONS]]
          .rename(columns=lambda c: c[3:]).round(3))

In [ ]:
# Detailed report + confusion matrix for the strongest classical model (by validation macro-F1)
best_key = max(PIPES, key=lambda k: f1_score(y_va, PIPES[k].predict(X_va), average="macro"))
print("Best classical model:", best_key)
pred = PIPES[best_key].predict(X_te)
print(classification_report(y_te, pred, target_names=EMOTIONS, digits=3, zero_division=0))
plot_cm(y_te, pred, f"Confusion matrix — {best_key[0]} ({best_key[1]})",
        path=ROOT / f"results/figures/02_cm_{best_key[0]}_{best_key[1]}.png")

## 4 · Interpretability — which terms drive each emotion? (Logistic Regression)
This is the main advantage of the linear baseline (paper Sec. 5.3).

In [ ]:
lr = PIPES[("logreg", "baseline")]
terms = np.array(lr.named_steps["tfidf"].get_feature_names_out())
coef = lr.named_steps["clf"].coef_
top = {ID2LABEL[i]: ", ".join(terms[np.argsort(coef[i])[-10:][::-1]]) for i in range(len(EMOTIONS))}
pd.Series(top, name="top-10 terms").to_frame()

### ✅ Done
Artifacts: `models/classical/*.joblib`, `results/model_metrics.csv`, `results/predictions_*.csv`.
`logreg_baseline.joblib` is also the **fallback predictor** used by the temporal notebook if no transformer is trained yet.
Next: **`03_bilstm.ipynb`**.